In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq
from pydantic import BaseModel
from typing import Annotated, TypedDict

llm = ChatGroq(model="openai/gpt-oss-20b")

class ChatState(TypedDict):
    messages: Annotated[list, add_messages]

def chat_bot(state: ChatState):
   
    response = llm.invoke(state["messages"])
    
    
    return {"messages": [response]}

graph = StateGraph(ChatState)

graph.add_node("chat_bot", chat_bot)
graph.add_edge(START, "chat_bot")
graph.add_edge("chat_bot", END)

memory = InMemorySaver()
final = graph.compile(checkpointer=memory)

res = final.invoke(
    {"messages": [HumanMessage(content="whats his age in 2026")]},
    {"configurable": {"thread_id": "id_1"}}
)


ans = res["messages"][-1].content
print(res)

{'messages': [HumanMessage(content='whats his age in 2026', additional_kwargs={}, response_metadata={}, id='033ebc5e-0133-4e01-b40c-4123fba84304'), AIMessage(content='I’m not sure who you’re referring to. Could you let me know who “he” is?', additional_kwargs={'reasoning_content': 'The user asks: "whats his age in 2026". We need to know who "he" refers to. The conversation context: The user previously asked "whats his age in 2026". There\'s no prior context in this conversation. The system message says we are ChatGPT. The user wants an answer. But we have no knowledge of who "he" is. We must ask for clarification. According to policy, we should ask clarifying question.'}, response_metadata={'token_usage': {'completion_tokens': 126, 'prompt_tokens': 79, 'total_tokens': 205, 'completion_time': 0.144727381, 'completion_tokens_details': {'reasoning_tokens': 96}, 'prompt_time': 0.004502355, 'prompt_tokens_details': None, 'queue_time': 0.284908826, 'total_time': 0.149229736}, 'model_name': '

In [10]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

connection = sqlite3.connect("db.sqlite3", check_same_thread=False)
sql_memory = SqliteSaver(connection)
sql_memory.setup()
sqlite_graph = graph.compile(checkpointer=sql_memory)
res = sqlite_graph.invoke(
        {"messages": 
            [ HumanMessage(content="Whats her age") ]
            },
        {"configurable": {"thread_id":"asd1"}}
    )
ans = res["messages"][-1].content
print(ans)

Nirmala Sitharaman was born on **24 April 1959**, so she is **67 years old** (as of 8 November 2026).
